In [6]:
from google.colab import files
uploaded = files.upload()  # A button will appear — upload spam.csv

Saving spam.csv to spam (1).csv


In [7]:
!pip install streamlit nltk scikit-learn joblib pyngrok -q

In [8]:
import pandas as pd
import nltk
import re
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# Load dataset
df = pd.read_csv('spam.csv', encoding='latin-1')
df.columns = ['label', 'message']

# Preprocess
stop_words = set(stopwords.words('english'))
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(w for w in text.split() if w not in stop_words)
    return text

df['cleaned']   = df['message'].apply(preprocess)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"✅ df is ready! Shape: {df.shape}")
print(df.head(3))

✅ df is ready! Shape: (5572, 4)
  label                                            message  \
0   ham  Go until jurong point, crazy.. Available only ...   
1   ham                      Ok lar... Joking wif u oni...   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...   

                                             cleaned  label_num  
0  go jurong point crazy available bugis n great ...          0  
1                            ok lar joking wif u oni          0  
2  free entry wkly comp win fa cup final tkts st ...          1  


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Vectorize
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned'])
y = df['label_num']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Train both models ─────────────────────────────────────────
models = {
    "Naive Bayes": MultinomialNB(),
    "SVM (LinearSVC)": LinearSVC()
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n{'='*40}")
    print(f"📊 {name} — Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# ── Pick the best model automatically ────────────────────────
best_name = max(results, key=results.get)
best_model = models[best_name]
print(f"\n🏆 Best model: {best_name} ({results[best_name]:.4f} accuracy)")

# ── Save the best model ───────────────────────────────────────
joblib.dump(best_model, 'model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print(f"✅ Saved best model ({best_name}) as model.pkl")


📊 Naive Bayes — Accuracy: 0.9767
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.99       966
        Spam       1.00      0.83      0.90       149

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.95      1115
weighted avg       0.98      0.98      0.98      1115


📊 SVM (LinearSVC) — Accuracy: 0.9883
              precision    recall  f1-score   support

         Ham       0.99      1.00      0.99       966
        Spam       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115


🏆 Best model: SVM (LinearSVC) (0.9883 accuracy)
✅ Saved best model (SVM (LinearSVC)) as model.pkl


In [10]:
from sklearn.svm import LinearSVC
import joblib

# Force SVM
model = LinearSVC()
model.fit(X_train, y_train)

joblib.dump(model,      'model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("✅ SVM model saved!")

✅ SVM model saved!


In [11]:
from google.colab import files
files.download('model.pkl')
files.download('vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>